In [4]:
import math
import random

# ==========================================
# 1. PROBLEM DATA (1 Depot + 8 Customers)
# ==========================================
# Node 0 is the Depot; Nodes 1 to 8 are Customers
COORDS = [
    (20, 20),  # 0: Depot
    (10, 10),  # 1: Customer 1
    (10, 30),  # 2: Customer 2
    (30, 30),  # 3: Customer 3
    (30, 10),  # 4: Customer 4
    (15, 15),  # 5: Customer 5
    (25, 25),  # 6: Customer 6
    (10, 20),  # 7: Customer 7
    (15, 40)   # 8: Customer 8
]

# Demand for each node (Depot has 0 demand)
DEMANDS = [0, 15, 20, 25, 10, 15, 20, 10, 15]
CAPACITY = 45
NUM_CUSTOMERS = len(COORDS) - 1


# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def distance(node1, node2):
    """Calculates Euclidean distance between two nodes."""
    x1, y1 = COORDS[node1]
    x2, y2 = COORDS[node2]
    return math.hypot(x2 - x1, y2 - y1)


def split_into_routes(customer_order):
    """Splits a single list of customers into valid vehicle routes."""
    routes = []
    current_route = [0]  # Start at Depot
    current_load = 0

    for cust in customer_order:
        if current_load + DEMANDS[cust] <= CAPACITY:
            current_route.append(cust)
            current_load += DEMANDS[cust]
        else:
            current_route.append(0)  # Return to Depot
            routes.append(current_route)
            # Start new vehicle route
            current_route = [0, cust]
            current_load = DEMANDS[cust]

    current_route.append(0)  # Final return to Depot
    routes.append(current_route)
    return routes


def total_distance(customer_order):
    """Fitness Function: Total distance of all vehicle routes (lower is better)."""
    routes = split_into_routes(customer_order)
    dist = 0.0
    for route in routes:
        for i in range(len(route) - 1):
            dist += distance(route[i], route[i + 1])
    return dist


# ==========================================
# 3. CUCKOO SEARCH OPERATORS
# ==========================================
def levy_mutation(order):
    """
    Simulates a Levy Flight on a list:
    - Mostly makes small local changes (swap 2 customers)
    - Occasionally makes a big jump (reverse a section of the route)
    """
    new_order = order[:]
    step = random.paretovariate(1.5)  # Heavy-tailed random step

    if step < 2.0:
        # Small step: Swap two random customers
        i, j = random.sample(range(len(new_order)), 2)
        new_order[i], new_order[j] = new_order[j], new_order[i]
    else:
        # Large Levy jump: Reverse a random sub-route (2-opt move)
        i, j = sorted(random.sample(range(len(new_order)), 2))
        new_order[i:j+1] = reversed(new_order[i:j+1])

    return new_order


def cuckoo_search(num_nests=20, abandon_rate=0.25, generations=300):
    customers = list(range(1, NUM_CUSTOMERS + 1))

    # Step 1: Create initial random solutions (nests)
    nests = [random.sample(customers, len(customers)) for _ in range(num_nests)]
    fitness = [total_distance(nest) for nest in nests]

    # Track the best solution found so far
    best_nest = nests[0][:]
    best_dist = fitness[0]

    # Step 2: Main Evolution Loop
    for gen in range(generations):
        # Pick a random nest and generate a new cuckoo solution via Levy flight
        i = random.randint(0, num_nests - 1)
        new_nest = levy_mutation(nests[i])
        new_dist = total_distance(new_nest)

        # Pick another random nest j and replace it if the new solution is better
        j = random.randint(0, num_nests - 1)
        if new_dist < fitness[j]:
            nests[j] = new_nest
            fitness[j] = new_dist

        # Abandon worst nests (Fraction Pa) and replace with new random ones
        num_abandon = int(abandon_rate * num_nests)
        # Pair fitness with nests, sort by distance, and replace the worst ones
        paired = sorted(zip(fitness, nests))
        for k in range(num_nests - num_abandon, num_nests):
            rand_nest = random.sample(customers, len(customers))
            paired[k] = (total_distance(rand_nest), rand_nest)

        # Unpack sorted nests back
        fitness = [p[0] for p in paired]
        nests = [p[1] for p in paired]

        # Update global best
        if fitness[0] < best_dist:
            best_dist = fitness[0]
            best_nest = nests[0][:]

    return split_into_routes(best_nest), best_dist


# ==========================================
# 4. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    random.seed(42)
    best_routes, min_dist = cuckoo_search()

    print(f"Minimum Total Distance: {min_dist:.2f}\n")
    for idx, route in enumerate(best_routes, 1):
        load = sum(DEMANDS[node] for node in route)
        print(f"Vehicle {idx}: {route}  (Load: {load}/{CAPACITY})")


Minimum Total Distance: 128.36

Vehicle 1: [0, 8, 2, 7, 0]  (Load: 45/45)
Vehicle 2: [0, 3, 6, 0]  (Load: 45/45)
Vehicle 3: [0, 4, 1, 5, 0]  (Load: 40/45)
